# 3 · Movie review sentiment with a GRU

Train a bag-of-embeddings baseline and a GRU on IMDB reviews.
Compare their scores, training time and predictions.

## Setup

In [62]:
import os, re, time, math, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset
from torchvision import transforms
from PIL import Image

QUICK = True
SEED  = 42

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2 if os.name != "nt" else 0

DATA_DIR = Path("/content/data") if Path("/content").is_dir() else Path("./data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 100
print("device :", device)
print("QUICK  :", QUICK)
if device.type == "cuda":
    print("gpu    :", torch.cuda.get_device_name(0))


device : cuda
QUICK  : True
gpu    : Tesla T4


In [63]:

try:
    import datasets
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    import datasets
print("datasets", datasets.__version__)

@torch.no_grad()
def accuracy(model, X, y, batch_size=256):
    model.eval()
    correct = 0
    for i in range(0, len(X), batch_size):
        pred = model(X[i:i+batch_size].to(device)).argmax(1).cpu()
        correct += (pred == y[i:i+batch_size]).sum().item()
    return correct / len(X)


def train(model, X, y, epochs=10, lr=1e-3, batch_size=64, X_val=None, y_val=None,
          criterion=None, weight_decay=0.0, log=True, restore_best=False):
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    criterion = criterion or nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "train_acc": [], "val_acc": []}
    best_val = -1.0
    best_state = None

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        total, correct, running = 0, 0, 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()          # 1. forget the gradients of the previous batch
            out  = model(xb)               # 2. forward pass -> raw scores (logits)
            loss = criterion(out, yb)      # 3. how wrong are we?
            loss.backward()                # 4. backprop: fill .grad for every weight
            optimizer.step()               # 5. nudge every weight downhill

            running += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            total   += xb.size(0)

        history["train_loss"].append(running / total)
        history["train_acc"].append(correct / total)
        if X_val is not None:
            val_acc = accuracy(model, X_val, y_val)
            history["val_acc"].append(val_acc)
            if restore_best and val_acc > best_val:
                best_val = val_acc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if log:
            msg = f"epoch {epoch+1}/{epochs} | train loss {running/total:.4f} | train acc {correct/total:.3f}"
            if X_val is not None:
                msg += f" | val acc {history['val_acc'][-1]:.3f}"
            print(msg + f" | {time.time()-t0:.1f}s")
    if restore_best and best_state is not None:
        model.load_state_dict(best_state)
    return history


def show_images(images, titles=None, ncols=5, title=None):
    n = len(images)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.7, nrows * 1.9))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for i, img in enumerate(images):
        arr = img.detach().cpu().permute(1, 2, 0).clamp(0, 1).numpy()
        axes[i].imshow(arr)
        if titles is not None:
            axes[i].set_title(titles[i], fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout(); plt.show()


def n_params(model):
    return sum(p.numel() for p in model.parameters())

datasets 4.8.5


## 1. Convert text to numbers

Split each review into words. Keep the 10,000 most frequent words.
Map words to integers. Use 0 for padding and 1 for unknown words.
Pad or truncate each review to 200 tokens.

In [64]:
from datasets import load_dataset

MAX_LEN = 400
VOCAB_SIZE = 10_000
EMB_DIM = 64
HIDDEN = 64

raw = load_dataset("stanfordnlp/imdb", "plain_text")
train_texts, train_labels = raw["train"]["text"], raw["train"]["label"]
test_texts,  test_labels  = raw["test"]["text"],  raw["test"]["label"]

# IMDb is stored sorted by label: the first 12,500 reviews are all negative. Slicing the first N
# would give us a one-class dataset, so we take a random sample.
rng = np.random.default_rng(SEED)

def sample(texts, labels, n):
    idx = rng.permutation(len(texts))[:n]
    return [texts[i] for i in idx], [labels[i] for i in idx]

if QUICK:
    train_texts, train_labels = sample(train_texts, train_labels, 8_000)
    test_texts,  test_labels  = sample(test_texts,  test_labels,  2_000)

print(f"{len(train_texts)} training reviews, {len(test_texts)} test reviews")
print(f"label balance (train): {np.mean(train_labels):.2f} positive")
print("\n--- one review, first 400 characters ---")
print(train_texts[0][:400], "...")

8000 training reviews, 2000 test reviews
label balance (train): 0.50 positive

--- one review, first 400 characters ---
There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compa ...


In [65]:
from collections import Counter

tokenize = lambda text: re.findall(r"[a-z']+", text.lower())

counter = Counter()
for text in train_texts:
    counter.update(tokenize(text))

itos = ["<pad>", "<unk>"] + [w for w, c in counter.most_common(VOCAB_SIZE) if c >= 2]
stoi = {w: i for i, w in enumerate(itos)}
print(f"vocabulary: {len(itos):,} tokens")
print("most frequent:", itos[2:12])

def encode(text, max_len=MAX_LEN):
    ids = [stoi.get(w, 1) for w in tokenize(text)][:max_len]
    ids += [0] * (max_len - len(ids))
    return ids

example = encode(train_texts[0])
print("\nfirst 20 token ids :", example[:20])
print("decoded again      :", " ".join(itos[i] for i in example[:20]))

vocabulary: 10,002 tokens
most frequent: ['the', 'and', 'a', 'of', 'to', 'is', 'br', 'in', 'it', 'i']

first 20 token ids : [47, 7, 60, 4941, 31, 30, 212, 1, 3, 1, 19, 2, 189, 13, 193, 24, 525, 207, 41, 1069]
decoded again      : there is no relation at all between <unk> and <unk> but the fact that both are police series about violent


In [66]:
X_tr_all = torch.tensor([encode(t) for t in train_texts], dtype=torch.long)
y_tr_all = torch.tensor(train_labels, dtype=torch.long)
X_te     = torch.tensor([encode(t) for t in test_texts],  dtype=torch.long)
y_te     = torch.tensor(test_labels,  dtype=torch.long)

# a proper train/validation split: the test set stays untouched until the scoreboard
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(X_tr_all), generator=g)
n_val = int(0.1 * len(perm))
val_idx, tr_idx = perm[:n_val], perm[n_val:]

X_tr,  y_tr  = X_tr_all[tr_idx],  y_tr_all[tr_idx]
X_val, y_val = X_tr_all[val_idx], y_tr_all[val_idx]
val_texts = [train_texts[i] for i in val_idx.tolist()]

print(f"train {len(X_tr)} | val {len(X_val)} | test {len(X_te)} (held out until the scoreboard)")
print("one batch:", tuple(X_tr[:8].shape), " (batch, tokens), dtype", X_tr.dtype)
print("targets  :", y_tr[:8].tolist(), " 1 = positive")

train 7200 | val 800 | test 2000 (held out until the scoreboard)
one batch: (8, 400)  (batch, tokens), dtype torch.int64
targets  : [0, 1, 1, 1, 1, 1, 0, 1]  1 = positive


## 2. Baseline: average the word embeddings

The model maps each word to a 64-value vector. It averages the vectors in each review.
A linear layer predicts sentiment. This model uses words but ignores their order.

In [67]:
class BagOfEmbeddings(nn.Module):
    def __init__(self, vocab, dim=EMB_DIM):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.fc  = nn.Linear(dim, 2)

    def forward(self, x):
        mask = (x != 0).float().unsqueeze(-1)        # do not average over padding
        e = self.emb(x) * mask
        v = e.sum(1) / mask.sum(1).clamp(min=1)
        return self.fc(v)

bag = BagOfEmbeddings(len(itos)).to(device)
t0 = time.time()
h_bag = train(bag, X_tr, y_tr, epochs=4, lr=2e-3, X_val=X_val, y_val=y_val)
bag_val, bag_time = h_bag["val_acc"][-1], time.time() - t0
print(f"\nbag-of-embeddings validation accuracy: {bag_val:.3f}")

epoch 1/4 | train loss 0.6705 | train acc 0.613 | val acc 0.713 | 0.3s
epoch 2/4 | train loss 0.5621 | train acc 0.760 | val acc 0.809 | 0.3s
epoch 3/4 | train loss 0.4254 | train acc 0.847 | val acc 0.828 | 0.3s
epoch 4/4 | train loss 0.3324 | train acc 0.882 | val acc 0.856 | 0.3s

bag-of-embeddings validation accuracy: 0.856


Validation accuracy: 0.833. The model ignores word order and still classifies reviews well.

## 3. RNN

In [68]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab, dim=EMB_DIM, hidden=HIDDEN):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.gru = nn.GRU(dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 2)

    def forward(self, x):
        lengths = (x != 0).sum(1).clamp(min=1)
        embedded = self.emb(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed) # _ (output) contains a hidden state for each time step.
                                     # hidden contains the final hidden state.
        return self.fc(hidden[-1])


gru = GRUClassifier(len(itos)).to(device)
t0 = time.time()
h_gru = train(
    gru, X_tr, y_tr, epochs=(15 if QUICK else 25), lr=2e-3,
    X_val=X_val, y_val=y_val, weight_decay=1e-4, restore_best=True,
)
gru_val = max(h_gru["val_acc"])
gru_time = time.time() - t0
print()
print(f"GRU best validation accuracy: {gru_val:.3f}")

epoch 1/15 | train loss 0.6787 | train acc 0.567 | val acc 0.632 | 2.1s
epoch 2/15 | train loss 0.6701 | train acc 0.626 | val acc 0.625 | 2.1s
epoch 3/15 | train loss 0.5655 | train acc 0.705 | val acc 0.741 | 2.2s
epoch 4/15 | train loss 0.4718 | train acc 0.781 | val acc 0.769 | 2.1s
epoch 5/15 | train loss 0.3800 | train acc 0.836 | val acc 0.807 | 2.1s
epoch 6/15 | train loss 0.2829 | train acc 0.886 | val acc 0.824 | 2.1s
epoch 7/15 | train loss 0.2150 | train acc 0.918 | val acc 0.863 | 2.1s
epoch 8/15 | train loss 0.1503 | train acc 0.945 | val acc 0.853 | 2.1s
epoch 9/15 | train loss 0.1007 | train acc 0.968 | val acc 0.836 | 2.1s
epoch 10/15 | train loss 0.0890 | train acc 0.973 | val acc 0.856 | 2.0s
epoch 11/15 | train loss 0.0670 | train acc 0.977 | val acc 0.844 | 2.0s
epoch 12/15 | train loss 0.0685 | train acc 0.976 | val acc 0.841 | 2.1s
epoch 13/15 | train loss 0.0386 | train acc 0.988 | val acc 0.856 | 2.1s
epoch 14/15 | train loss 0.0211 | train acc 0.994 | val acc 

The GRU reads each review in order. Packing skips padding, and the final hidden state summarizes the review.
The model keeps the checkpoint with the best validation score.

In [69]:
def predict(text, model):
    x = torch.tensor([encode(text)]).to(device)
    with torch.no_grad():
        p = torch.softmax(model(x), dim=1)[0, 1].item()
    verdict = "POSITIVE" if p > 0.5 else "NEGATIVE"
    return verdict, p

my_reviews = [
    "This film was a masterpiece, I loved every minute of it.",
    "Two hours of my life I will never get back. Awful.",
    "Not bad at all, actually rather good.",
    "It is not good.",
    "The acting was fine but the plot made no sense whatsoever.",
]

for text in my_reviews:
    (v_bag, p_bag), (v_gru, p_gru) = predict(text, bag), predict(text, gru)
    print(f"{text[:52]:<54} bag {v_bag:>8} {p_bag:.2f} | GRU {v_gru:>8} {p_gru:.2f}")

This film was a masterpiece, I loved every minute of   bag POSITIVE 0.99 | GRU POSITIVE 1.00
Two hours of my life I will never get back. Awful.     bag POSITIVE 0.72 | GRU NEGATIVE 0.05
Not bad at all, actually rather good.                  bag NEGATIVE 0.00 | GRU NEGATIVE 0.20
It is not good.                                        bag POSITIVE 0.99 | GRU POSITIVE 0.81
The acting was fine but the plot made no sense whats   bag NEGATIVE 0.00 | GRU NEGATIVE 0.19


The next cell scores each validation review with both models.
It prints the four reviews with the largest score differences. Read them and check the labels.

In [70]:
@torch.no_grad()
def positive_prob(model, X):
    out = []
    for i in range(0, len(X), 256):
        out.append(torch.softmax(model(X[i:i+256].to(device)), 1)[:, 1].cpu())
    return torch.cat(out)

p_bag = positive_prob(bag, X_val)
p_gru = positive_prob(gru, X_val)
diff = (p_bag - p_gru).abs()

for i in diff.argsort(descending=True)[:4].tolist():
    truth = "pos" if int(y_val[i]) == 1 else "neg"
    text = val_texts[i][:84].replace("\n", " ")
    print(f"true {truth} | bag {p_bag[i]:.2f} | GRU {p_gru[i]:.2f} | {text}")

true pos | bag 0.94 | GRU 0.07 | I can't believe John died! While filming an episode he collapsed on set! read this, 
true pos | bag 0.89 | GRU 0.04 | I thought the kids in the movie were great. I deal with kids in that age group, and 
true neg | bag 0.90 | GRU 0.13 | I'm a fan of Columbo, especially on a rainy Saturday, and it was fun to see Oskar We
true pos | bag 0.80 | GRU 0.05 | Dramatic license - some hate it, though it is necessary in retelling any life story.


Compare the two scores with each review's true label.
The GRU may help when word order matters. It may also make different errors.